In [28]:
# 1. CELL: Könyvtárak betöltése és alapbeállítások
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Stílusbeállítások
plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_palette("viridis")

print("✅ Könyvtárak betöltve!")

✅ Könyvtárak betöltve!


In [30]:
# 2. CELL: Adatok betöltése és előkészítése
try:
    df = pd.read_csv("zenga_rentals_details_optimized.csv")
    df['price'] = df['price'] * 1000
    print(f"📊 Adatok betöltve: {len(df)} albérleti hirdetés")
    
    # Alapvető tisztítás
    initial_count = len(df)
    df = df[df['price'].notna() & df['area_m2'].notna()]
    df = df[(df['price'] > 50_000) & (df['price'] < 1_000_000)]  # realis berletek (eFt)
    df = df[(df['area_m2'] > 15) & (df['area_m2'] < 300)]    # Reális méretek
    
    # Price per m² kiszámolása
    df['price_per_m2'] = df['price'] / df['area_m2']
      
    # Kerület kinyerése location-ból
    def extract_district(location):
        if pd.isna(location):
            return None
        location_str = str(location)
        for i in range(1, 24):
            if f"{i}." in location_str or f"{i}. kerület" in location_str:
                return i
        return None
    
    df['kerület'] = df['location'].apply(extract_district)
    
    print(f"✅ Tisztított adatok: {len(df)} hirdetés ({initial_count - len(df)} eltávolítva)")
except FileNotFoundError:
    print("❌ Fájl nem található! Futtasd először a adatgyűjtőt.")

📊 Adatok betöltve: 1762 albérleti hirdetés
✅ Tisztított adatok: 1443 hirdetés (319 eltávolítva)


In [31]:
# 3. CELL: Statisztikai elemzés - Legjobb ajánlatok automatikus azonosítása
def find_statistical_best_deals(df, n_deals=10):
    """Statisztikai módszerekkel megtalálja a legjobb ajánlatokat"""
    if df is None:
        return None
    
    print("📈 Statisztikai elemzés futtatása...")
    
    # Ár per m² alapú értékelés
    df_analysis = df.copy()
    
    # 1. Z-score alapú outlier detektálás (túl olcsó/jó ajánlatok)
    df_analysis['price_zscore'] = np.abs(stats.zscore(df_analysis['price_per_m2'].fillna(
        df_analysis['price_per_m2'].median())))
    
    # 2. Kerületi átlagok számítása
    district_stats = df_analysis.groupby('kerület').agg({
        'price_per_m2': ['mean', 'std'],
        'price': 'mean'
    }).round(0)
    district_stats.columns = ['district_avg_price_m2', 'district_std_price_m2', 'district_avg_price']
    
    # 3. Relatív érték számítás (ár/átlag arány)
    df_analysis = df_analysis.merge(district_stats, on='kerület', how='left')
    df_analysis['price_ratio'] = df_analysis['price_per_m2'] / df_analysis['district_avg_price_m2']
    
    # 4. Értékelési pontszám számítása
    df_analysis['value_score'] = (
        (1 / df_analysis['price_ratio']) * 0.6 +  # Ár/érték arány (60%)
        (1 / df_analysis['price_zscore']) * 0.4    # Statisztikai normalitás (40%)
    )
    
    # 5. Legjobb ajánlatok kiválasztása
    best_deals = df_analysis.nlargest(n_deals, 'value_score')[[
        'title', 'price', 'area_m2', 'price_per_m2', 'kerület', 
        'location', 'value_score', 'price_ratio'
    ]]
    
    best_deals = best_deals.round({'price_per_m2': 0, 'value_score': 3, 'price_ratio': 2})
    best_deals = best_deals.rename(columns={
        'price_ratio': 'ár_arány',  # 1.0 = átlag, <1.0 = jobb mint átlag
        'value_score': 'érték_pont'
    })
    
    return best_deals

# Legjobb ajánlatok megjelenítése
if df is not None:
    best_deals = find_statistical_best_deals(df, 15)
    print("🏆 TOP 15 legjobb értékű ajánlat (statisztikai alapon):")
    display(best_deals)

📈 Statisztikai elemzés futtatása...
🏆 TOP 15 legjobb értékű ajánlat (statisztikai alapon):


,title,price,area_m2,price_per_m2,kerület,location,érték_pont,ár_arány
468,Budapest V. kerület kiadó téglalakás 2 szobás:...,420000.0,66.0,6364.0,5.0,"Budapest V. kerület, Lipótváros, Deák tér köze...",5.365,1.12
131,Budapest IX. kerület kiadó téglalakás 2 szobás...,320000.0,50.0,6400.0,2.0,"Budapest IX. kerület, Belső Ferencváros, Vaska...",4.529,0.98
24,Budapest VII. kerület kiadó téglalakás 1+1 fél...,209000.0,35.0,5971.0,8.0,"Budapest VII. kerület, Ligetváros, István utca...",3.785,1.06
254,Budapest VII. kerület kiadó téglalakás 3 szobá...,400000.0,71.0,5634.0,9.0,"Budapest VII. kerület, Belső-Erzsébetváros, Ka...",2.108,0.76
20,Budapest VII. kerület kiadó téglalakás 1+1 fél...,250000.0,44.0,5682.0,10.0,"Budapest VII. kerület, Külső-Erzsébetváros, Jó...",1.881,1.37
454,Budapest V. kerület kiadó téglalakás 1+1 fél s...,350000.0,51.0,6863.0,6.0,"Budapest V. kerület, Lipótváros, Október 6. u.",1.829,0.89
675,Budapest II. kerület kiadó téglalakás 3 szobás...,375000.0,68.0,5515.0,2.0,"Budapest II. kerület, Törökvész, 2.ker. Bimbó ...",1.803,0.85
808,Budapest XIII. kerület kiadó téglalakás 2+1 fé...,270000.0,83.0,3253.0,6.0,"Budapest XIII. kerület, Újlipótváros, Váci út 6.",1.681,0.42
640,Budapest XIV. kerület kiadó téglalakás 2 szobá...,320000.0,60.0,5333.0,8.0,"Budapest XIV. kerület, Istvánmező, Ilka utca 58.",1.503,0.94
383,Budapest VI. kerület kiadó téglalakás 2 szobás...,496000.0,69.0,7188.0,6.0,"Budapest VI. kerület, 6. kerület, Ó utca",1.416,0.93


In [35]:
# 4. CELL: Interaktív szűrőrendszer létrehozása
def create_rental_filter_widget(df):
    """Interaktív szűrőwidget létrehozása"""
    
    # Elérhető kerületek
    available_districts = sorted([d for d in df['kerület'].unique() if not pd.isna(d)])
    
    # Widgetek definiálása
    price_range = widgets.IntRangeSlider(
        value=[df['price'].min(), df['price'].max()],
        min=df['price'].min(),
        max=df['price'].max(),
        step=5000,
        description='Ár (eFt/hó):',
        continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    area_range = widgets.IntRangeSlider(
        value=[df['area_m2'].min(), df['area_m2'].max()],
        min=df['area_m2'].min(),
        max=df['area_m2'].max(),
        step=5,
        description='Terület (m²):',
        continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    rooms_widget = widgets.SelectMultiple(
        options=sorted(df['rooms'].dropna().unique()),
        value=tuple(sorted(df['rooms'].dropna().unique())),
        description='Szobák:',
        style={'description_width': 'initial'}
    )
    
    districts_widget = widgets.SelectMultiple(
        options=available_districts,
        value=tuple(available_districts),
        description='Kerületek:',
        style={'description_width': 'initial'}
    )
    
    # Szűrési gomb
    filter_button = widgets.Button(
        description="🔍 Szűrés alkalmazása",
        button_style='primary'
    )
    
    # Eredménytábla
    result_output = widgets.Output()
    
    def apply_filters(b):
        with result_output:
            result_output.clear_output()
            
            # Szűrés alkalmazása
            filtered_df = df.copy()
            
            # Ár szűrés
            filtered_df = filtered_df[
                (filtered_df['price'] >= price_range.value[0]) & 
                (filtered_df['price'] <= price_range.value[1])
            ]
            
            # Terület szűrés
            filtered_df = filtered_df[
                (filtered_df['area_m2'] >= area_range.value[0]) & 
                (filtered_df['area_m2'] <= area_range.value[1])
            ]
            
            # Szobák szűrés
            if rooms_widget.value:
                filtered_df = filtered_df[filtered_df['rooms'].isin(rooms_widget.value)]
            
            # Kerület szűrés
            if districts_widget.value:
                filtered_df = filtered_df[filtered_df['kerület'].isin(districts_widget.value)]
            
            # Rendezés ár szerint
            filtered_df = filtered_df.sort_values('price')
            
            print(f"✅ {len(filtered_df)} találat a szűrési feltételek alapján")
            print("=" * 80)
            
            if len(filtered_df) > 0:
                # Fontos oszlopok kiválasztása
                display_cols = ['title', 'price', 'area_m2', 'price_per_m2', 'kerület', 'location']
                display_df = filtered_df[display_cols].head(20)
                display_df['price_per_m2'] = display_df['price_per_m2'].round(0)
                
                # Formázás
                styled_df = display_df.style\
                    .format({
                        'price': '{:,.0f} Ft'.format,
                        'price_per_m2': '{:,.0f} Ft/m²'.format,
                        'area_m2': '{:.0f} m²'.format
                    })\
                    .background_gradient(subset=['price_per_m2'], cmap='RdYlGn_r')\
                    .background_gradient(subset=['price'], cmap='viridis')
                
                display(styled_df)
                
                # Statisztikák
                print(f"\n📊 Statisztikák:")
                print(f"   • Átlagár: {filtered_df['price'].mean():,.0f} Ft/hó")
                print(f"   • Átlagos ár/m²: {filtered_df['price_per_m2'].mean():,.0f} Ft/m²")
                print(f"   • Legolcsóbb: {filtered_df['price'].min():,.0f} Ft/hó")
                print(f"   • Legdrágább: {filtered_df['price'].max():,.0f} Ft/hó")
            else:
                print("❌ Nincs találat a megadott szűrési feltételeknek!")
    
    filter_button.on_click(apply_filters)
    
    # UI elrendezés
    filters_box = widgets.VBox([
        widgets.HTML("<h3>🔍 Szűrési feltételek</h3>"),
        price_range,
        area_range,
        rooms_widget,
        districts_widget,
        filter_button
    ])
    
    return widgets.VBox([
        widgets.HTML("<h2>🏠 Albérlet Kereső - Testreszabott Szűrés</h2>"),
        widgets.HBox([filters_box, result_output])
    ])

# Widget megjelenítése
if df is not None:
    rental_filter = create_rental_filter_widget(df)
    display(rental_filter)

In [36]:
# 5. CELL: Speciális keresés speciális tulajdonságokra
def create_advanced_filter_widget(df):
    """Speciális szűrők speciális tulajdonságokra"""
    
    # Elérhető tulajdonságok
    available_conditions = df['Állapot'].dropna().unique()
    available_heating = df['Fűtés'].dropna().unique()
    available_energy = df['Energetikai besorolás'].dropna().unique()
    
    # Widgetek
    condition_widget = widgets.SelectMultiple(
        options=available_conditions,
        description='Állapot:',
        style={'description_width': 'initial'}
    )
    
    heating_widget = widgets.SelectMultiple(
        options=available_heating,
        description='Fűtés:',
        style={'description_width': 'initial'}
    )
    
    energy_widget = widgets.SelectMultiple(
        options=available_energy,
        description='Energetika:',
        style={'description_width': 'initial'}
    )
    
    balcony_widget = widgets.Checkbox(
        value=False,
        description='Csak erkélyes lakások',
        style={'description_width': 'initial'}
    )
    
    terrace_widget = widgets.Checkbox(
        value=False,
        description='Csak teraszes lakások',
        style={'description_width': 'initial'}
    )
    
    search_button = widgets.Button(
        description="🔎 Keresés",
        button_style='success'
    )
    
    result_output = widgets.Output()
    
    def advanced_search(b):
        with result_output:
            result_output.clear_output()
            
            filtered_df = df.copy()
            
            # Speciális szűrések
            if condition_widget.value:
                filtered_df = filtered_df[filtered_df['Állapot'].isin(condition_widget.value)]
            
            if heating_widget.value:
                filtered_df = filtered_df[filtered_df['Fűtés'].isin(heating_widget.value)]
            
            if energy_widget.value:
                filtered_df = filtered_df[filtered_df['Energetikai besorolás'].isin(energy_widget.value)]
            
            if balcony_widget.value:
                filtered_df = filtered_df[filtered_df['has_balcony'] == 1]
            
            if terrace_widget.value:
                filtered_df = filtered_df[filtered_df['has_terrace'] == 1]
            
            # Rendezés
            filtered_df = filtered_df.sort_values('price')
            
            print(f"✅ {len(filtered_df)} találat speciális szűréssel")
            
            if len(filtered_df) > 0:
                display_cols = ['title', 'price', 'area_m2', 'price_per_m2', 'kerület', 
                               'Állapot', 'Fűtés', 'Energetikai besorolás']
                display_df = filtered_df[display_cols].head(15)
                
                styled_df = display_df.style\
                    .format({
                        'price': '{:,.0f} Ft'.format,
                        'price_per_m2': '{:,.0f} Ft/m²'.format,
                        'area_m2': '{:.0f} m²'.format
                    })\
                    .background_gradient(subset=['price'], cmap='RdYlGn_r')
                
                display(styled_df)
            else:
                print("❌ Nincs találat!")
    
    search_button.on_click(advanced_search)
    
    return widgets.VBox([
        widgets.HTML("<h3>🎯 Speciális Keresés</h3>"),
        condition_widget,
        heating_widget,
        energy_widget,
        balcony_widget,
        terrace_widget,
        search_button,
        result_output
    ])

# Speciális keresés megjelenítése
if df is not None:
    advanced_search_widget = create_advanced_filter_widget(df)
    display(advanced_search_widget)

In [37]:
# 6. CELL: Vizualizációk és elemzések
def create_visualizations(df):
    """Vizualizációk létrehozása az albérleti piac elemzéséhez"""
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Ár eloszlás kerületenként', 'Ár/m² eloszlás', 
                       'Ár és terület kapcsolata', 'Top 10 legjobb értékű kerület')
    )
    
    # 1. Ár eloszlás kerületenként
    district_prices = df.groupby('kerület')['price'].mean().sort_values(ascending=False)
    fig.add_trace(
        go.Bar(x=district_prices.index, y=district_prices.values, name='Átlagár'),
        row=1, col=1
    )
    
    # 2. Ár/m² eloszlás
    fig.add_trace(
        go.Box(y=df['price_per_m2'], name='Ár/m²', boxpoints=False),
        row=1, col=2
    )
    
    # 3. Ár és terület kapcsolata
    fig.add_trace(
        go.Scatter(x=df['area_m2'], y=df['price'], mode='markers', 
                  marker=dict(size=8, opacity=0.6), name='Lakások'),
        row=2, col=1
    )
    
    # 4. Legjobb értékű kerületek (ár/érték arány)
    district_value = df.groupby('kerület').apply(
        lambda x: x['price_per_m2'].mean() / x['area_m2'].mean()
    ).sort_values(ascending=True).head(10)
    
    fig.add_trace(
        go.Bar(x=district_value.values, y=district_value.index, 
               orientation='h', name='Érték arány'),
        row=2, col=2
    )
    
    fig.update_layout(height=800, showlegend=False, title_text="Albérlet Piac Elemzés")
    fig.show()
    
    # További statisztikák
    print("📈 Általános statisztikák:")
    print(f"   • Átlagos bérleti díj: {df['price'].mean():,.0f} Ft/hó")
    print(f"   • Átlagos ár/m²: {df['price_per_m2'].mean():,.0f} Ft/m²")
    print(f"   • Medián ár: {df['price'].median():,.0f} Ft/hó")
    print(f"   • Legolcsóbb kerület: {district_prices.idxmin()} ({district_prices.min():,.0f} Ft/hó)")
    print(f"   • Legdrágább kerület: {district_prices.idxmax()} ({district_prices.max():,.0f} Ft/hó)")

# Vizualizációk megjelenítése
if df is not None:
    create_visualizations(df)

C:\Users\Adam\AppData\Local\Temp\ipykernel_6288\416143426.py:32: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



📈 Általános statisztikák:
   • Átlagos bérleti díj: 398,860 Ft/hó
   • Átlagos ár/m²: 6,207 Ft/m²
   • Medián ár: 340,000 Ft/hó
   • Legolcsóbb kerület: 7.0 (160,000 Ft/hó)
   • Legdrágább kerület: 3.0 (911,000 Ft/hó)
